In [ ]:
# MATLAB section 1/4 for StimulusDecode2D: 2-D Stimulus Decode

# % 2-D Stimulus Decode
# Here we simulate hippocampal place cell receptive fields and their firing
# during a 2-d spatial task. We then use the ensemble firing activity to
# estimate the path based on the only the point process observations
#
# MATLAB: delta = 0.001;
# MATLAB: Tmax = 1;
# MATLAB: time = 0:delta:Tmax;
# MATLAB: px = zeros(1,length(time));
# MATLAB: py = zeros(1,length(time));
# MATLAB: Q=.01;
# MATLAB: r =  Q.*randn(2,length(time));
# MATLAB: vx = cumsum(r(1,:))';
# MATLAB: vy = cumsum(r(2,:))';
#
# MATLAB: velSig = SignalObj(time, [vx, vy],'vel');
# MATLAB: posSig = velSig.integral;
# MATLAB: posData = posSig.data;
# MATLAB: px = posData(:,1);
# MATLAB: py = posData(:,2);
# N=100; A=1; B=ones(1,N)./N;
# px = filtfilt(B,A,px);
# py = filtfilt(B,A,py);
# MATLAB: figure;
# MATLAB: plot(px,py);
# MATLAB: title('Simulated X-Y trajectory');
# MATLAB: xlabel('x'); ylabel('y');
#
#

# Python translation bootstrap for this helpfile.

# Topic: StimulusDecode2D
# Execution group: full
# Workflow family: decoding_2d
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/StimulusDecode2D.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "StimulusDecode2D"
FAMILY = "decoding_2d"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"StimulusDecode2D: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"StimulusDecode2D: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"StimulusDecode2D: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"StimulusDecode2D: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 7

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)

import os
from pathlib import Path
MATLAB_HELP_ROOT = next((p for p in [
    Path(os.environ['NSTAT_MATLAB_HELP_ROOT']) if os.environ.get('NSTAT_MATLAB_HELP_ROOT') else None,
    Path('/tmp/upstream-nstat/helpfiles'),
    Path('/private/tmp/upstream-nstat/helpfiles'),
] if p is not None and p.exists()), None)

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=1, matlab_line_number=24, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "StimulusDecode2D.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 1, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()


In [ ]:
# MATLAB section 2/4 for StimulusDecode2D: Generate random receptive fields to simulate different neurons

# % Generate random receptive fields to simulate different neurons
# MATLAB: clear lambdaCIF lambda tempSpikeColl n spikeColl
# MATLAB: numRealizations=80;
#
# MATLAB: coeffs = -abs(1*randn(numRealizations,5));
# MATLAB: coeffs = [-2*abs(randn(numRealizations,1)) coeffs];
# MATLAB: dataMat = [ones(length(time),1) px py px.^2 py.^2 px.*py];
# MATLAB:  for i=1:numRealizations
# MATLAB:      tempData  = exp(dataMat*coeffs(i,:)');
# MATLAB:      lambdaData = tempData./(1+tempData);
# MATLAB:      lambda{i}=Covariate(time,lambdaData./delta, '\Lambda(t)','time','s','Hz',{strcat('\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});
#
# MATLAB:      tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1);
# MATLAB:      n{i} = tempSpikeColl{i}.getNST(1);
# MATLAB:      n{i}.setName(num2str(i));
#
# MATLAB:      try
# MATLAB:          lambdaCIF{i} = CIF(coeffs(i,:),{'1','x','y','x^2','y^2','x*y'},{'x','y'},'binomial');
# MATLAB:      catch ME_sym
# MATLAB:          if(i==1)
# MATLAB:              warning('StimulusDecode2D:SymbolicCIFFallback', ...
# MATLAB:                  ['CIF symbolic setup failed (' ME_sym.identifier '). Decoder will use linear fallback.']);
# MATLAB:          end
# MATLAB:          lambdaCIF{i} = [];
# MATLAB:      end
# MATLAB:  end
#
#
# View the different neuron conditional intensity functions
# MATLAB:  figure;
# MATLAB:  for i=1:length(lambda)
# MATLAB:     lambda{i}.plot;
# MATLAB:  end
# MATLAB:  legend off;
#
# Visualize Simulated Receptive Fields
# MATLAB: clear placeField;
# MATLAB: [X,Y]=meshgrid(-2:.1:2,-2:.1:2);
# MATLAB: figure;
#
# MATLAB: for i=1:numRealizations
# MATLAB: tempData = coeffs(i,1) + coeffs(i,2)*X + coeffs(i,3)*Y +coeffs(i,4)*X.^2 + coeffs(i,5)*Y.^2 + coeffs(i,6).*X.*Y;
# MATLAB: placeField{i} = exp(tempData)./(1+exp(tempData))./delta; %rate based on logistic link function
#
# MATLAB: end
#
# MATLAB: fact=factor(numRealizations);
#
# MATLAB: for i=1:numRealizations
# MATLAB:    if(length(fact)==1)
# MATLAB:     subplot(1,numRealizations,i);
# MATLAB:    elseif(length(fact)==2)
# MATLAB:     subplot(fact(1),fact(2),i);
# MATLAB:    elseif(length(fact)==3)
# MATLAB:     subplot(fact(1)*fact(2),fact(3),i);
# MATLAB:    end
# MATLAB:     pcolor(X,Y,placeField{i}), shading interp
# MATLAB:     axis square;
# MATLAB:     set(gca,'xtick',[],'ytick',[]);
#
# MATLAB: end

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=2, matlab_line_number=59, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "StimulusDecode2D_01.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 2, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=2, matlab_line_number=68, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "StimulusDecode2D_02.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 2, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/4 for StimulusDecode2D: Decode the x-y trajectory

# % Decode the x-y trajectory
#
# MATLAB:  spikeColl = nstColl(n);
# MATLAB:  spikeColl.resample(1/delta);
# MATLAB:  dN = spikeColl.dataToMatrix;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/4 for StimulusDecode2D: Section

# %
# MATLAB: vx=var(px(2:end)-px(1:end-1));
# MATLAB: vy=var(py(2:end)-py(1:end-1));
# MATLAB: Q=[vx 0;0 vy];
# MATLAB: Px0=.1*eye(2,2); A=1*eye(2,2);
# MATLAB: decode_method = 'PPDecodeFilter';
# MATLAB: try
# MATLAB:     [x_p, Pe_p, x_u, Pe_u] = DecodingAlgorithms.PPDecodeFilter(A, Q, Px0, dN',lambdaCIF,delta);
# MATLAB: catch ME_decode
# MATLAB:     warning('StimulusDecode2D:SymbolicDecodeFallback', ...
# MATLAB:         ['PPDecodeFilter failed (' ME_decode.identifier '). Falling back to PPDecodeFilterLinear.']);
# MATLAB:     decode_method = 'PPDecodeFilterLinear';
# MATLAB:     mu_linear = coeffs(:,1);
# MATLAB:     beta_linear = coeffs(:,2:3)';
# MATLAB:     [x_p, Pe_p, x_u, Pe_u] = DecodingAlgorithms.PPDecodeFilterLinear(A, Q, dN', mu_linear, beta_linear, 'binomial', delta);
# MATLAB: end
# MATLAB: nCommon = min(length(px),size(x_u,2));
# MATLAB: decode_rmse = sqrt(mean((x_u(1,1:nCommon)'-px(1:nCommon)).^2 + (x_u(2,1:nCommon)'-py(1:nCommon)).^2));
# MATLAB: num_cells = numRealizations;
# MATLAB: figure;
# MATLAB: plot(x_u(1,:),x_u(2,:),'b',px,py,'k')
# MATLAB: legend('predicted path','actual path');
#
# Parity contract scalars for MATLAB/Python verification.
# MATLAB: parity = struct();
# MATLAB: parity.num_cells = num_cells;
# MATLAB: parity.decode_rmse = decode_rmse;
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=116, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "StimulusDecode2D_03.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=4 + 4, title=f"{TOPIC} Figure 004")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #5 for StimulusDecode2D>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=97, matlab_snippet="<synthetic MATLAB figure event #5 for StimulusDecode2D>")
ref_image = (MATLAB_HELP_ROOT / "StimulusDecode2D_04.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=5 + 4, title=f"{TOPIC} Figure 005")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #6 for StimulusDecode2D>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=97, matlab_snippet="<synthetic MATLAB figure event #6 for StimulusDecode2D>")
ref_image = (MATLAB_HELP_ROOT / "StimulusDecode2D_05.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=6 + 4, title=f"{TOPIC} Figure 006")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #7 for StimulusDecode2D>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=97, matlab_snippet="<synthetic MATLAB figure event #7 for StimulusDecode2D>")
ref_image = (MATLAB_HELP_ROOT / "StimulusDecode2D_06.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=7 + 4, title=f"{TOPIC} Figure 007")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "delta = 0.001;",
    "Tmax = 1;",
    "time = 0:delta:Tmax;",
    "px = zeros(1,length(time));",
    "py = zeros(1,length(time));",
    "Q=.01;",
    "r =  Q.*randn(2,length(time));",
    "vx = cumsum(r(1,:))';",
    "vy = cumsum(r(2,:))';",
    "velSig = SignalObj(time, [vx, vy],'vel');",
    "posSig = velSig.integral;",
    "posData = posSig.data;",
    "px = posData(:,1);",
    "py = posData(:,2);",
    "figure;",
    "plot(px,py);",
    "title('Simulated X-Y trajectory');",
    "xlabel('x'); ylabel('y');",
    "clear lambdaCIF lambda tempSpikeColl n spikeColl",
    "numRealizations=80;",
    "coeffs = -abs(1*randn(numRealizations,5));",
    "coeffs = [-2*abs(randn(numRealizations,1)) coeffs];",
    "dataMat = [ones(length(time),1) px py px.^2 py.^2 px.*py];",
    "for i=1:numRealizations",
    "tempData  = exp(dataMat*coeffs(i,:)');",
    "lambdaData = tempData./(1+tempData);",
    "lambda{i}=Covariate(time,lambdaData./delta, '\\Lambda(t)','time','s','Hz',{strcat('\\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});",
    "tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1);",
    "n{i} = tempSpikeColl{i}.getNST(1);",
    "n{i}.setName(num2str(i));",
    "try",
    "lambdaCIF{i} = CIF(coeffs(i,:),{'1','x','y','x^2','y^2','x*y'},{'x','y'},'binomial');",
    "catch ME_sym",
    "if(i==1)",
    "warning('StimulusDecode2D:SymbolicCIFFallback', ...",
    "['CIF symbolic setup failed (' ME_sym.identifier '). Decoder will use linear fallback.']);",
    "end",
    "lambdaCIF{i} = [];",
    "end",
    "end",
    "figure;",
    "for i=1:length(lambda)",
    "lambda{i}.plot;",
    "end",
    "legend off;",
    "clear placeField;",
    "[X,Y]=meshgrid(-2:.1:2,-2:.1:2);",
    "figure;",
    "for i=1:numRealizations",
    "tempData = coeffs(i,1) + coeffs(i,2)*X + coeffs(i,3)*Y +coeffs(i,4)*X.^2 + coeffs(i,5)*Y.^2 + coeffs(i,6).*X.*Y;",
    "placeField{i} = exp(tempData)./(1+exp(tempData))./delta; %rate based on logistic link function",
    "end",
    "fact=factor(numRealizations);",
    "for i=1:numRealizations",
    "if(length(fact)==1)",
    "subplot(1,numRealizations,i);",
    "elseif(length(fact)==2)",
    "subplot(fact(1),fact(2),i);",
    "elseif(length(fact)==3)",
    "subplot(fact(1)*fact(2),fact(3),i);",
    "end",
    "pcolor(X,Y,placeField{i}), shading interp",
    "axis square;",
    "set(gca,'xtick',[],'ytick',[]);",
    "end",
    "spikeColl = nstColl(n);",
    "spikeColl.resample(1/delta);",
    "dN = spikeColl.dataToMatrix;",
    "vx=var(px(2:end)-px(1:end-1));",
    "vy=var(py(2:end)-py(1:end-1));",
    "Q=[vx 0;0 vy];",
    "Px0=.1*eye(2,2); A=1*eye(2,2);",
    "decode_method = 'PPDecodeFilter';",
    "try",
    "[x_p, Pe_p, x_u, Pe_u] = DecodingAlgorithms.PPDecodeFilter(A, Q, Px0, dN',lambdaCIF,delta);",
    "catch ME_decode",
    "warning('StimulusDecode2D:SymbolicDecodeFallback', ...",
    "['PPDecodeFilter failed (' ME_decode.identifier '). Falling back to PPDecodeFilterLinear.']);",
    "decode_method = 'PPDecodeFilterLinear';",
    "mu_linear = coeffs(:,1);",
    "beta_linear = coeffs(:,2:3)';",
    "[x_p, Pe_p, x_u, Pe_u] = DecodingAlgorithms.PPDecodeFilterLinear(A, Q, dN', mu_linear, beta_linear, 'binomial', delta);",
    "end",
    "nCommon = min(length(px),size(x_u,2));",
    "decode_rmse = sqrt(mean((x_u(1,1:nCommon)'-px(1:nCommon)).^2 + (x_u(2,1:nCommon)'-py(1:nCommon)).^2));",
    "num_cells = numRealizations;",
    "figure;",
    "plot(x_u(1,:),x_u(2,:),'b',px,py,'k')",
    "legend('predicted path','actual path');",
    "parity = struct();",
    "parity.num_cells = num_cells;",
    "parity.decode_rmse = decode_rmse;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for StimulusDecode2D.")

# StimulusDecode2D: fixture-backed 2D trajectory decoding parity check.
from pathlib import Path
import nstat
from scipy.io import loadmat
fixture_path = Path(nstat.__file__).resolve().parents[2] / "tests/parity/fixtures/matlab_gold/StimulusDecode2D_gold.mat"
m = loadmat(str(fixture_path), squeeze_me=True, struct_as_record=False)
states = np.asarray(m["states_sd"], dtype=float); latent = np.asarray(m["latent_sd"], dtype=int).reshape(-1)
tuning = np.asarray(m["tuning_sd"], dtype=float); spike_counts = np.asarray(m["spike_counts_sd"], dtype=float)
decoded_center = DecodingAlgorithms.decode_weighted_center(spike_counts=spike_counts, tuning_curves=tuning)
decoded = np.clip(np.rint(decoded_center), 0, states.shape[0] - 1).astype(int)
xy_true = np.asarray(m["xy_true_sd"], dtype=float); xy_decoded = states[decoded]
rmse = float(np.sqrt(np.mean(np.sum((xy_decoded - xy_true) ** 2, axis=1))))
expected_center = np.asarray(m["decoded_center_sd"], dtype=float).reshape(-1); expected_decoded = np.asarray(m["decoded_sd"], dtype=int).reshape(-1); expected_rmse = float(np.asarray(m["rmse_sd"], dtype=float).reshape(-1)[0])
center_err = float(np.max(np.abs(decoded_center - expected_center))); decoded_mismatch = float(np.count_nonzero(decoded != expected_decoded)); rmse_err = float(abs(rmse - expected_rmse))
assert center_err <= 1e-8 and decoded_mismatch == 0.0 and rmse_err <= 1e-10

side = int(round(np.sqrt(states.shape[0]))); field_idx = 3
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))
axes[0].plot(xy_true[:, 0], xy_true[:, 1], label="true", linewidth=1.2)
axes[0].plot(xy_decoded[:, 0], xy_decoded[:, 1], label="decoded", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: decoded trajectory"); axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].set_aspect("equal", adjustable="box"); axes[0].legend(loc="upper right")
im = axes[1].imshow(tuning[field_idx].reshape(side, side), origin="lower", extent=[0.0, 1.0, 0.0, 1.0], cmap="jet", aspect="equal")
axes[1].set_title("Example receptive field"); axes[1].set_xlabel("x"); axes[1].set_ylabel("y"); fig.colorbar(im, ax=axes[1], fraction=0.04, pad=0.03)
plt.tight_layout(); plt.show()

CHECKPOINT_METRICS = {"trajectory_rmse": float(rmse), "decoded_unique_states": float(np.unique(decoded).size), "decoded_center_max_abs_error": center_err, "decoded_mismatch_count": decoded_mismatch}
CHECKPOINT_LIMITS = {"trajectory_rmse": (0.0, 1.5), "decoded_unique_states": (2.0, float(states.shape[0])), "decoded_center_max_abs_error": (0.0, 1e-8), "decoded_mismatch_count": (0.0, 0.0)}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
